# Straight-line versus quadratic-Bezier extension — validation only

This notebook clones the public repository without a token, checks out exact implementation commit `7bdcd2e847ca7c5a1faf8a086b26441d8de1a4e1`, runs the complete test suite, and creates the no-output validation report. It does not generate comparative paintings, authorize execution, train a model, mount Drive, or modify any closed experiment. Run all cells in order on a standard CPU runtime.

In [ ]:
from pathlib import Path
import shutil
import subprocess

REPO_DIR = Path('/content/latent-stroke-dynamics')
REPO_URL = 'https://github.com/Navid111/latent-stroke-dynamics.git'
BRANCH = 'quadratic-bezier-extension'
EXPECTED_COMMIT = '7bdcd2e847ca7c5a1faf8a086b26441d8de1a4e1'

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
completed = subprocess.run(
    ['git', 'clone', '--quiet', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)],
    capture_output=True,
    text=True,
)
if completed.returncode != 0:
    print(completed.stdout)
    print(completed.stderr)
    raise RuntimeError('Public repository clone failed.')
subprocess.run(['git', 'checkout', '--quiet', '--detach', EXPECTED_COMMIT], cwd=REPO_DIR, check=True)
observed_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
assert observed_commit == EXPECTED_COMMIT, observed_commit
assert subprocess.check_output(['git', 'status', '--short'], cwd=REPO_DIR, text=True).strip() == ''
print('CELL 1 COMPLETE — PUBLIC REPOSITORY CLONED; EXACT SOURCE PINNED', observed_commit)

In [ ]:
import subprocess
from pathlib import Path

subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.'], cwd=REPO_DIR, check=True)
test_run = subprocess.run(
    ['python', '-m', 'pytest', '-q'],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
)
PYTEST_LOG = Path('/content/quadratic_bezier_pytest.txt')
PYTEST_LOG.write_text(test_run.stdout + test_run.stderr, encoding='utf-8')
print(test_run.stdout)
if test_run.stderr:
    print(test_run.stderr)
assert test_run.returncode == 0, 'The complete test suite failed.'
print('CELL 2 COMPLETE — COMPLETE TEST SUITE PASSED')

In [ ]:
import json
import subprocess
from pathlib import Path

validation_run = subprocess.run(
    ['python', 'validate_quadratic_bezier_extension.py', '--validate-only'],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
)
if validation_run.returncode != 0:
    print(validation_run.stdout)
    print(validation_run.stderr)
    raise RuntimeError('Validation-only command failed.')
VALIDATION_JSON = Path('/content/quadratic_bezier_validation.json')
VALIDATION_JSON.write_text(validation_run.stdout, encoding='utf-8')
validation = json.loads(validation_run.stdout)
print('CELL 3 COMPLETE — VALIDATION REPORT CREATED IN /content ONLY')

In [ ]:
import subprocess

assert validation['status'] == 'quadratic_bezier_extension_valid_no_outputs'
assert validation['protocol_id'] == 'quadratic_bezier_extension_v1'
assert validation['output_side_effects'] is False
assert validation['comparative_outputs_viewed'] is False
assert validation['training_performed'] is False
assert validation['learned_model_used'] is False
assert validation['execution_authorized'] is False
assert validation['closed_experiments_changed'] is False
assert validation['proposed_target_manifest']['target_count'] == 6
assert validation['protocol']['target_hashes_frozen'] is False
assert validation['protocol']['completed_executions'] == 0
assert set(validation['synthetic_smoke']) == {'straight', 'quadratic_bezier'}
assert all(item['deterministic'] for item in validation['synthetic_smoke'].values())
assert all(item['monotonic'] for item in validation['synthetic_smoke'].values())
worktree_status = subprocess.check_output(['git', 'status', '--short'], cwd=REPO_DIR, text=True).strip()
assert worktree_status == '', worktree_status
print('CELL 4 COMPLETE — VALIDATION-ONLY GATE PASSED')
print('status:', validation['status'])
print('target-set SHA-256:', validation['proposed_target_manifest']['target_set_sha256'])
for item in validation['proposed_target_manifest']['targets']:
    print(item['target_id'], item['pixel_sha256'])
print('straight smoke:', validation['synthetic_smoke']['straight'])
print('quadratic smoke:', validation['synthetic_smoke']['quadratic_bezier'])
print('output side effects:', validation['output_side_effects'])
print('execution authorized:', validation['execution_authorized'])
print('git status clean:', worktree_status == '')

In [ ]:
from google.colab import files

files.download(str(PYTEST_LOG))
files.download(str(VALIDATION_JSON))
print('CELL 5 COMPLETE — DOWNLOADS STARTED')